# 08 · Cache & Storage Levels em um Cluster Real

**Teoria**: docs/06-persistencia-e-otimizacao.md

**Pré-requisito**: `make up-cluster` ainda em execução.

🎯 **Objetivo**: explorar os diferentes níveis de persistência do Spark (`MEMORY_ONLY` vs 
`MEMORY_AND_DISK`) e o impacto do **Adaptive Query Execution (AQE)** em dados distorcidos 
(data skew).

In [ ]:
import sys
import time

sys.path.insert(0, "../scripts")
from lab_utils import get_connect_session, layer_path
from pyspark import StorageLevel

# Sessão Spark Connect — processamento remoto no cluster Docker
# make up-cluster deve estar ativo
spark = get_connect_session("07-cache-storage-levels")

# Lê o dataset de vendas (centenas de milhares de linhas) dentro do container
vendas = spark.read.parquet(layer_path("connect", "bronze", "vendas"))

## `MEMORY_ONLY` vs. `MEMORY_AND_DISK`

🧠 **Conceito**: `persist()` permite que você escolha exatamente onde os dados em cache residem. 
Verifique a aba **Storage** da Spark UI (http://localhost:4040/storage/) após cada célula para ver 
o DataFrame realmente materializado lá, com seu tamanho e fração de 
armazenamento.

| Nível | Descrição | Se não couber na RAM |
|---|---|---|
| `MEMORY_ONLY` | Apenas RAM | Recomputa do lineage (caro!) |
| `MEMORY_AND_DISK` | RAM + SSD | Derrama em disco (mais seguro) |

💡 **Dica**: `MEMORY_ONLY` é mais rápido se o dataset couber na RAM. 
`MEMORY_AND_DISK` é mais tolerante a picos de memória.

In [ ]:
# Persiste em MEMORY_ONLY: dados ficam apenas na RAM dos executores
# Se não couber, partições excedentes são RECOMPUTADAS do lineage quando necessário
vendas.persist(StorageLevel.MEMORY_ONLY)
vendas.count()  # Materializa o cache — força a leitura e o armazenamento em memória
print("Cached with MEMORY_ONLY — check the Storage tab now.")

📌 **Verificação**:

Na aba **Storage** da Spark UI (http://localhost:4040/storage/), você deve ver 
o DataFrame `vendas` listado com:
- **Size in Memory**: tamanho ocupado na RAM
- **Size in Disk**: 0 B (pois é MEMORY_ONLY)
- **Fraction Cached**: fração do dataset que coube na RAM

Se `Fraction Cached < 1`, parte dos dados teria que ser recomputada se acessada novamente.

In [ ]:
# Limpa o cache anterior antes de aplicar novo StorageLevel
vendas.unpersist()

# Persiste em MEMORY_AND_DISK: RAM primeiro, SSD como fallback
# Se não couber na RAM, o Spark derrama o excedente em disco (mais seguro)
vendas.persist(StorageLevel.MEMORY_AND_DISK)
vendas.count()
print("Cached with MEMORY_AND_DISK — check the Storage tab again.")
vendas.unpersist()

📌 **Comparando os dois níveis**:

Na aba **Storage**, observe a diferença:
- Com `MEMORY_ONLY`: se `Fraction Cached < 1`, dados parciais podem ser perdidos
- Com `MEMORY_AND_DISK`: mesmo que não caiba tudo na RAM, o disco evita perda

💡 **Qual escolher?**
  - Dados que cabem na RAM → `MEMORY_ONLY` (mais rápido)
  - Dados grandes ou imprevisíveis → `MEMORY_AND_DISK` (mais seguro)
  - Cache entre stages de shuffle → `MEMORY_AND_DISK_SER` (serializado, mais compacto)

## Adaptive Query Execution (AQE) e data skew

🧠 **Problema**: dados distorcidos (skew) — uma partição muito maior que as outras — 
faz uma Task demorar muito mais, atrasando o Job inteiro (gargalo).

🎯 **Experimente**: construímos uma fatia deliberadamente distorcida — 90% das linhas caem 
em uma única região (`Sudeste`) — depois agregamos com AQE ligado e desligado.

📌 Observe a aba **Stages** da Spark UI: com AQE ligado, o Spark divide a partition 
distorcida em sub-tarefas menores (Skew Join Optimization) em vez de deixar uma única 
Task fazer 90% do trabalho sozinha.

In [ ]:
from pyspark.sql.functions import rand, when

# Cria uma coluna com skew artificial: 90% de chance de cair em "Sudeste"
# seed=42 garante reprodutibilidade — mesmos números aleatórios toda execução
skewed = vendas.withColumn(
    "regiao_skewed",
    when(rand(seed=42) < 0.9, "Sudeste").otherwise(vendas.regiao),
)
skewed.createOrReplaceTempView("vendas_skewed")

# Executa a mesma consulta com AQE ligado e desligado para comparar
for aqe in (False, True):
    # Alterna a configuração do AQE dinamicamente — sem precisar reiniciar a sessão
    spark.conf.set("spark.sql.adaptive.enabled", str(aqe).lower())
    start = time.perf_counter()
    spark.sql(
        "SELECT regiao_skewed, SUM(valor) FROM vendas_skewed GROUP BY regiao_skewed"
    ).collect()
    elapsed = time.perf_counter() - start
    print(f"AQE={aqe!s:5s} -> {elapsed:.2f}s (see Spark UI Stages tab for task-level detail)")

# Restaura o padrão — AQE ligado (recomendado para a maioria dos workloads)
spark.conf.set("spark.sql.adaptive.enabled", "true")

📌 **Análise do AQE**:

Com AQE **desligado**, o Spark executa o plano estático: todas as Tasks processam 
volumes iguais de partições — mas como uma partição é muito maior (devido ao skew), 
aquela Task específica demora muito mais (straggler).

Com AQE **ligado**, o Spark monitora o runtime e:
1. Detecta partições distorcidas (skew)
2. Divide a partição grande em sub-partições menores (skew join/skew group optimization)
3. Balanceia a carga entre as Tasks disponíveis

💡 **Dica**: AQE está habilitado por padrão no Spark 3.4+. Desligá-lo pode fazer sentido 
apenas em cenários muito específicos (ex.: dados perfeitamente uniformes).

In [ ]:
# Encerra a sessão Spark Connect — libera recursos no cluster Docker
spark.stop()